<a href="https://colab.research.google.com/github/aa6910/BPhO-Computational-Challenge-2024/blob/main/2_3_gpus_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to GPUs in PyTorch
GPUs were initially designed for graphics (GPU stands for "Graphics Processing Unit").  The kind of hardware that you need for graphics turns out to massively speed up deep learning / AI.  

Specifically, in graphics, you need to compute the color of each pixel about 50 times a second.  And you typically have about a million pixels on screen.  That's 50 million pixels per second.  Which is huge.  GPUs solve this problem by having many, many small cores all computing pixel colours in parallel.  Critically, GPUs simplify their operations by forcing each of the small cores to do the same operations (because computing the colour of the pixel in the top-left is a very similar operation to computing the pixel in the bottom right).  This is called a "single-instruction, multiple-data" or SIMD architecture.  And it turns out that this kind of architecture is exactly what we need to massively speed up large tensor operations in Deep Learning!

Typically you're looking at 50-100 times speedups, for large tensor operations.  Which is huge.  In fact, the "deep learning revolution" only really got started when deep networks were accelerated by putting them on GPUs [[AlexNet paper]](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf).

In this notebook, we won't get all the way to deep networks.  Instead, we'll explore how PyTorch can be used as a "GPU enabled Numpy".  That means you can use GPUs in PyTorch to accelerate all kinds of scientific programming: not just in deep learning/AI!




















## Running this notebook

This notebook requires access to a GPU.  Colab doesn't give you a GPU by default.  To get a GPU in Colab, go to:
* "Runtime" (in the menu along with "File", "Edit" etc.)
* "Change runtime type"
* Under "Hardward Accelerator" change "None" to "GPU"

In [ ]:
#Import PyTorch
import torch as t

In [ ]:
#Make a tensor on the GPU (the device='cuda')
a = t.rand(3, device='cuda')
a

tensor([0.8901, 0.5346, 0.7572], device='cuda:0')

In [ ]:
#You can do all the usual things
print(2*a)
print(a.exp())
print(a*a)

tensor([1.7803, 1.0693, 1.5145], device='cuda:0')
tensor([2.4355, 1.7068, 2.1324], device='cuda:0')
tensor([0.7924, 0.2858, 0.5734], device='cuda:0')


In [ ]:
#And if you make a second tensor on the GPU, you can combine them:
b = t.rand(3, device='cuda')
print(b)
print(a+b)

tensor([0.9633, 0.1554, 0.7820], device='cuda:0')
tensor([1.8534, 0.6901, 1.5393], device='cuda:0')


In [ ]:
#However, if you don't use `device='cuda'`, the tensor defaults to the CPU,
#and tensors need to be on the same device for you to be able to combine them
c = t.rand(3)
a+c

RuntimeError: ignored

In [ ]:
#To check the device of a tensor, use .device
print(a.device)
print(b.device)
print(c.device)

cuda:0
cuda:0
cpu


In [ ]:
#You can move a pre-existing tensor to the GPU using .to(device='cuda')
c_cuda = c.to(device='cuda')
#Note that this doesn't move the original tensor, which remains unchanged and on the CPU,
print(c.device)
#It gives you a new tensor on the GPU,
print(c_cuda.device)
#Which can be combined with other tensors on the GPU
print(a+c_cuda)

cpu
cuda:0
tensor([1.1323, 1.0858, 1.5905], device='cuda:0')


In [ ]:
del a
del b
del c
del c_cuda

## Why bother with GPUs?  Because they can be _wildly_ faster than CPUs

GPUs are massively parallel single-instruction multiple-data (SIMD) processors.  Like a giant CPU with many cores, but where all the cores have to do the same thing.  That means GPUs can give _massive_ speedups when doing operations on big tensors (e.g. matrix multiply).  But they tend not to give any speedups for doing lots of operations on small tensors.

Here, we see that the GPU is about 50 times faster than CPU for operations on big Tensors, but is similar, and maybe even a bit slower when doing lots of little operations on smaller tensors!



In [ ]:
a_cpu  = t.randn(1000, 1000, 30)
a_cuda = a_cpu.to(device='cuda')

In [ ]:
# Time a big operation on the CPU
%%timeit
(2*a_cpu).mean()

77.5 ms ± 7.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
# Time the same operation on the GPU
%%timeit
(2*a_cuda).mean()

1.45 ms ± 527 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
#But they aren't any faster if you're doing lots of small operations
a_cpu  = t.randn(3, 3, 30)
a_cuda = a_cpu.to(device='cuda')

In [ ]:
# Time a lots of little operations on the CPU
%%timeit
for _ in range(100):
    (2*a_cpu).mean()

1.68 ms ± 829 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
# Time a lots of little operations on the GPU
%%timeit
for _ in range(100):
    (2*a_cuda).mean()

3.79 ms ± 603 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
del a_cpu
del a_cuda

## Gotcha 1: Initializing on GPU vs moving to GPU

It can be quite slow to:
* Create tensors on the CPU (especially if they are e.g. random tensors).
* Move big tensors from CPU to GPU.  

If possible, it is always better to initialize/generate tensors directly on the GPU.


In [ ]:
#Its fast to generate noise on GPU directly,
%%timeit
t.randn(1000, 1000, 30, device='cuda').mean()

1.04 ms ± 926 ns per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
#Its much, much slower to generate noise on CPU, then transfer to GPU
%%timeit
t.randn(1000, 1000, 30).cuda().mean()

260 ms ± 12.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
#There are two components to this extra cost.
#First, its slower to generate random numbers on CPU.
%%timeit
t.randn(1000, 1000, 30)

339 ms ± 65.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
a = t.randn(1000, 1000, 30)

In [ ]:
%%timeit
a.cuda()

26.2 ms ± 1.03 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
del a

## Gotcha 2: GPU memory

In [ ]:
#One of the key issues you might end-up hitting is that GPUs have very limited memory
#If you try to allocate a _really big_ tensor directly on the GPU, we'll get this error.
#Note, if you allocate to the CPU memory / RAM first, you'll probably crash your CoLab
#instance, so don't do that...
a = t.randn(1000, 1000, 1000, 30, device='cuda')

OutOfMemoryError: ignored

In [ ]:
#To check the amount of GPU memory currently allocated, run t.cuda.memory_allocated(0).
#It returns the amount of memory, measured in bytes.
#We divide by 1024**3 to get a measurement in gigabytes.
t.cuda.memory_allocated(0)/1024**3

4.76837158203125e-07

In [ ]:
#If we allocate a big tensor, we see we have allocated more memory.
#Typically, we have about 16GB total with the GPUs on CoLab, so this is a
#big chunk of the available memory.
a = t.randn(1000, 1000, 1000, device='cuda')
t.cuda.memory_allocated(0)/1024**3
#Note that this tensor is about 1 billion elements.  And by default, PyTorch uses
#float32 numbers, which each require 4 bytes (1 byte = 8 bits).  So total memory
#consumption should be about 4 billion bytes.  And it is

3.7252907752990723

In [ ]:
#We can release the memory by deleting a,
del a

In [ ]:
#So we have more memory available again.
#(Note that the allocation only seems to go down if you make this call in a
#separate code block, as releasing the memory is a bit of a complex process,
#using lots of Python internals)
t.cuda.memory_allocated(0)/1024**3

4.76837158203125e-07

In [ ]:
#When running big neural-nets in for-loops, it is very easy to accidentally
#"leak" memory, so you eventually run out.

#For instance, lets say we're trying to compute the means for a bunch of
#big random tensors.  We'll print the memory at each step of the for-loop,
#so we can see the memory running out!

big_random_tensors=[]
means=[]
for i in range(20):
    print(t.cuda.memory_allocated(0)/1024**3)
    big_random_tensors.append(t.randn(1000, 1000, 500, device='cuda'))
    means.append(big_random_tensors[i].mean())

4.76837158203125e-07
1.8626461029052734
3.7252917289733887
5.587937355041504
7.450582981109619
9.313864707946777
11.177146434783936
13.040428161621094


OutOfMemoryError: ignored

In [ ]:
#The problem here is that we're saving each of these big tensors for future use.
print(big_random_tensors[2].shape)

torch.Size([1000, 1000, 500])


In [ ]:
#But we only really care about the means.
#Lets get rid of everything and try again,
del big_random_tensors
del means

In [ ]:
#And check the memory has been freed
t.cuda.memory_allocated(0)/1024**3

4.76837158203125e-07

In [ ]:
#Lets avoid saving a list of big_random_tensors, instead, just save the means
means=[]
for i in range(20):
    print(t.cuda.memory_allocated(0)/1024**3)
    big_random_tensor = t.randn(1000, 1000, 500, device='cuda')
    means.append(big_random_tensor.mean())

4.76837158203125e-07
1.8632822036743164
1.8632826805114746
1.8632831573486328
1.863283634185791
1.8632841110229492
1.8632845878601074
1.8632850646972656
1.8632855415344238
1.863286018371582
1.8632864952087402
1.8632869720458984
1.8632874488830566
1.8632879257202148
1.863288402557373
1.8632888793945312
1.8632893562316895
1.8632898330688477
1.8632903099060059
1.863290786743164


In [ ]:
#The memory use is constant!
#That's because we can't access big_random_tensor from previous loop iteraions,
#so the memory is "garbage collected".

#But there seems to be something left-over:
t.cuda.memory_allocated(0)/1024**3

1.8632912635803223

In [ ]:
#What's happened?  Well it turns out that we can still access big_random_tensor,
#so that memory hasn't been garbage collected,
print(big_random_tensor.shape)

torch.Size([1000, 1000, 500])


In [ ]:
#Lets delete that,
del big_random_tensor

In [ ]:
#check the memory has been freed, and try again
t.cuda.memory_allocated(0)/1024**3

1.0013580322265625e-05

In [ ]:
#How do we avoid that "hangover" in the first place?
#We need to somehow make big_random_tensor inaccessible outside the loop.
#One option would be to explicitly delete big_random_tensor.  But this is
#inelegant (you basically never see explicit `del` operations in real code).
#Instead, we can run a function inside the loop,

means=[]
def inner():
    big_random_tensor = t.randn(1000, 1000, 500, device='cuda')
    means.append(big_random_tensor.mean())

for i in range(20):
    print(t.cuda.memory_allocated(0)/1024**3)
    inner()

1.0013580322265625e-05
1.049041748046875e-05
1.0967254638671875e-05
1.1444091796875e-05
1.1920928955078125e-05
1.239776611328125e-05
1.2874603271484375e-05
1.33514404296875e-05
1.3828277587890625e-05
1.430511474609375e-05
1.4781951904296875e-05
1.52587890625e-05
1.5735626220703125e-05
1.621246337890625e-05
1.6689300537109375e-05
1.71661376953125e-05
1.7642974853515625e-05
1.811981201171875e-05
1.8596649169921875e-05
1.9073486328125e-05


In [ ]:
#Now, big_random_tensor cannot be accessed (as it is only defined inside the
#inner function), and so all the memory allocated has been freed,
t.cuda.memory_allocated(0)/1024**3

1.0013580322265625e-05

## Pytorch reference and GC subtleties

In [ ]:
#Allocate a small tensor
a = t.ones(3)
#We can update values in that tensor,
a[0] = 2
a

tensor([2., 1., 1.])

In [ ]:
#Now, make a second reference to the same tensor
b = a
#If we update b,
b[1] = 3
#Then we also update a
print(b)
print(a)
#Because both the names a and b point at the same underlying tensor.

tensor([2., 3., 1.])
tensor([2., 3., 1.])


In [ ]:
#This has implications for memory.
#Lets allocate a new tensor
a = t.randn(1000, 1000, 500, device='cuda')
t.cuda.memory_allocated(0)/1024**3

1.8632912635803223

In [ ]:
#Lets make a new reference to the same underlying tensor
b = a
#Its just a reference to the same underlying object, so memory usage is the same
t.cuda.memory_allocated(0)/1024**3

1.8632912635803223

In [ ]:
#Moreover, calling `del a` deletes the _reference_, not the underlying tensor.
del a
#So we can still access the tensor through the name b
b.shape

torch.Size([1000, 1000, 500])

In [ ]:
#And the memory hasn't been released.
t.cuda.memory_allocated(0)/1024**3

1.8632912635803223

In [ ]:
#To release the memory, we also need to call,
del b

In [ ]:
#Now, the underly tensor is inaccessible, so it will be garbage collected, and
#the memory freed
t.cuda.memory_allocated(0)/1024**3

1.0013580322265625e-05

## In-place operations
Note: you can't backprop through in-place operations.  So you can't use them e.g. inside a neural network.  But they are super-useful for actually doing gradient descent, or for for general scientific computing.

The basic idea is that when you do an operation, e.g.
\begin{align}
b_{ijk} = a_{ijk} + 100
\end{align}
You now have allocated two huge tensors, $a$ and $b$.
But quite often, once you've computed $b$, you don't need $a$.
In that case, you might want to update $a$ "in-place".

In PyTorch, there are a bunch of in-place updates.  There are two conventions for in-place updates.  Either they are methods with a trailing underscore,
```
x.add_(3)
```
Or you can use an explicit `out` keyword argument to indicate where you want to put a result.


In [ ]:
A = t.zeros(3,3)
B = t.randn(3,3)
C = t.randn(3,3)
t.matmul(B, C, out=A)
print(A)

tensor([[ 1.7164, -1.2915,  1.7376],
        [ 2.3750,  2.5088,  3.4997],
        [-0.0290, -4.0805, -1.3318]])


But the main use for in-place operations is saving GPU memory.

In [ ]:
#Lets say we want random Gaussian noise, with mean 100 and stddev 1.  We could do,
a = t.randn(1000, 1000, 500, device='cuda')
print(f"a[0,0,0] = {a[0,0,0].item()}")
b = a + 100
print(f"b[0,0,0] = {b[0,0,0].item()}")
#But that's not ideal because we've allocated two _huge_ tensors, a and b.
t.cuda.memory_allocated(0)/1024**3

a[0,0,0] = 0.5023083090782166
b[0,0,0] = 100.50231170654297


3.7265725135803223

In [ ]:
del a
del b

In [ ]:
t.cuda.memory_allocated(0)/1024**3

1.0013580322265625e-05

In [ ]:
#An alternative is to do an in-place operation
a = t.randn(1000, 1000, 500, device='cuda')
print(f"a[0,0,0] = {a[0,0,0].item()}")
a.add_(100)
print(f"a[0,0,0] = {a[0,0,0].item()}")
#Now, we've only allocated one giant tensor.  Doing the update in-place is also typically quite a bit faster
t.cuda.memory_allocated(0)/1024**3

a[0,0,0] = -1.1961995363235474
a[0,0,0] = 98.80380249023438


1.8632912635803223

In [ ]:
del a

In [ ]:
t.cuda.memory_allocated(0)/1024**3

1.0013580322265625e-05